# Clase 10 — Auxiliares · el gimnasio

8 ejercicios cortos sobre *El framework: Strategy + Backtest*: drills para ganar soltura con las primitivas y profundizaciones opcionales.

### El gimnasio

Drills cortos para automatizar las primitivas de Python con datos de mercado, más una profundización final. Mismo formato de siempre: escribe tu código, ejecuta la **✅ comprobación plegada** (`Shift+Enter`) y, si te atascas, abre **💡 Ver solución**. Ninguno debería llevarte más de un par de minutos. No hacen falta para seguir el curso — pero te hacen rápido.

**Dosis mínima:** el calentamiento entero + los dos primeros drills de cada bloque. El resto, para volver otro día.

---

## 🏋️ Gimnasio · Calentamiento — repaso exprés de L9

El loop y la fórmula de la cuenta, en dos reps.

### C1. El día exprés

¿Cuántos pasos tiene `Market.sample()`? Cuéntalos en `n` (o usa `len`).

<sub>practicas: repaso: contar pasos</sub>

In [ ]:
from exchange.market import Market
m = Market.sample()
n = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert n is not None, '⏸ n sigue en None: completa el ejercicio antes de validar'
assert n == 500
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
n = len(m)
```

</details>

### C2. equity exprés

Con `cash = -9999.0` y `position = 0.1`, calcula `eq` al mark 100000.

<sub>practicas: repaso: la fórmula</sub>

In [ ]:
cash = -9999.0
position = 0.1
eq = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert eq is not None, '⏸ eq sigue en None: completa el ejercicio antes de validar'
assert abs(eq - 1.0) < 1e-9
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
eq = cash + position * 100000
```

</details>

---

## 🏋️ Gimnasio · Bloque 1 — El contrato Strategy

Subclases, acciones y el runner: el enchufe, drill a drill.

### A1. La estrategia que no hace nada

Escribe `Hold(Strategy)` cuyo `on_book_update` devuelva `[]`. Pásala por el Backtest y guarda `result`.

<sub>practicas: el contrato mínimo</sub>

In [ ]:
from exchange.backtest import Backtest
from exchange.market import Market
from exchange.strategy import Strategy
result = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert result is not None, '⏸ result sigue en None: completa el ejercicio antes de validar'
assert result.n_steps == 500 and result.n_fills == 0
assert abs(result.final_equity) < 1e-9
print('ok ->', result)

<details>
<summary>💡 Ver solución</summary>

```python
class Hold(Strategy):
    def on_book_update(self, book):
        return []

result = Backtest(Market.sample(), Hold()).run()
```

</details>

### A2. Una acción de verdad

Construye la acción `a = NewOrder(Order('BTCUSDT', 'buy', 0.05, order_type=OrderType.MARKET))` y comprueba sus campos.

<sub>practicas: NewOrder</sub>

In [ ]:
from exchange.orders import Order, OrderType
from exchange.strategy import NewOrder
a = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert a is not None, '⏸ a sigue en None: completa el ejercicio antes de validar'
assert isinstance(a, NewOrder)
assert a.order.side == 'buy' and a.order.size == 0.05
print('ok ->', a.order)

<details>
<summary>💡 Ver solución</summary>

```python
a = NewOrder(Order('BTCUSDT', 'buy', 0.05, order_type=OrderType.MARKET))
```

</details>

### A3. BuyOnce, tuya

Escribe `BuyOnce(Strategy)`: en el primer libro devuelve UNA market buy de 0.1 y después siempre `[]`. Ejecútala y guarda `result`.

<sub>practicas: la subclase clásica</sub>

In [ ]:
from exchange.backtest import Backtest
from exchange.market import Market
from exchange.orders import Order, OrderType
from exchange.strategy import NewOrder, Strategy
result = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert result is not None, '⏸ result sigue en None: completa el ejercicio antes de validar'
assert result.n_fills >= 1
assert result.final_position > 0
print('ok ->', result)

<details>
<summary>💡 Ver solución</summary>

```python
class BuyOnce(Strategy):
    def __init__(self):
        self._done = False
    def on_book_update(self, book):
        if self._done:
            return []
        self._done = True
        return [NewOrder(Order('BTCUSDT', 'buy', 0.1, order_type=OrderType.MARKET))]

result = Backtest(Market.sample(), BuyOnce()).run()
```

</details>

### A4. El gancho on_fill

Añade a tu BuyOnce un contador `self.n_fills` que crezca en `on_fill`. Tras el run, debe coincidir con `result.n_fills`.

<sub>practicas: feedback del motor</sub>

In [ ]:
from exchange.backtest import Backtest
from exchange.market import Market
from exchange.orders import Order, OrderType
from exchange.strategy import NewOrder, Strategy
strat = None
result = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert strat is not None, '⏸ strat sigue en None: completa el ejercicio antes de validar'
assert result is not None, '⏸ result sigue en None: completa el ejercicio antes de validar'
assert strat.n_fills == result.n_fills
assert strat.n_fills >= 1
print('ok ->', strat.n_fills, 'fills contados por la estrategia')

<details>
<summary>💡 Ver solución</summary>

```python
class BuyOnce(Strategy):
    def __init__(self):
        self._done = False
        self.n_fills = 0
    def on_book_update(self, book):
        if self._done:
            return []
        self._done = True
        return [NewOrder(Order('BTCUSDT', 'buy', 0.1, order_type=OrderType.MARKET))]
    def on_fill(self, fill):
        self.n_fills += 1

strat = BuyOnce()
result = Backtest(Market.sample(), strat).run()
```

</details>

### A5. El mismo runner, dos vidas

Ejecuta `Hold` y `BuyOnce` (dadas) por el MISMO `Backtest.run()` y guarda `(eq_hold, eq_buy)`. Ni una línea del motor cambió.

<sub>practicas: polimorfismo en producción</sub>

In [ ]:
from exchange.backtest import Backtest
from exchange.market import Market
from exchange.orders import Order, OrderType
from exchange.strategy import NewOrder, Strategy

class Hold(Strategy):
    def on_book_update(self, book):
        return []

class BuyOnce(Strategy):
    def __init__(self):
        self._done = False
    def on_book_update(self, book):
        if self._done:
            return []
        self._done = True
        return [NewOrder(Order('BTCUSDT', 'buy', 0.1, order_type=OrderType.MARKET))]
eq_hold = None
eq_buy = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert eq_hold is not None, '⏸ eq_hold sigue en None: completa el ejercicio antes de validar'
assert eq_buy is not None, '⏸ eq_buy sigue en None: completa el ejercicio antes de validar'
assert abs(eq_hold) < 1e-9
assert abs(eq_buy) > 1e-9, 'BuyOnce tiene posicion: su equity se movio'
print(f'ok  hold={eq_hold} buy={eq_buy:.2f}')

<details>
<summary>💡 Ver solución</summary>

```python
eq_hold = Backtest(Market.sample(), Hold()).run().final_equity
eq_buy = Backtest(Market.sample(), BuyOnce()).run().final_equity
```

</details>

---

## 🏋️ Para terminar — profundización

Una estrategia con señal de verdad, lista para el juicio de L11.

### A6. Estrategia con señal

Define `ImbalanceBuyer`: compra 0.1 (market) solo cuando `book.imbalance(1) > 0.3`. Guarda `fills` del backtest.

<sub>practicas: decidir según el libro</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class ImbalanceBuyer(Strategy):
    def on_book_update(self, book):
        pass
fills = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert ImbalanceBuyer.on_book_update.__code__.co_consts != (None,), '⏸ implementa ImbalanceBuyer.on_book_update: su cuerpo sigue siendo pass'
assert fills is not None and fills >= 0
print('ok  fills=%d' % fills)

<details>
<summary>💡 Ver solución</summary>

```python
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class ImbalanceBuyer(Strategy):
    def on_book_update(self, book):
        if book.imbalance(1) is not None and book.imbalance(1) > 0.3:
            return [NewOrder(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET))]
        return []
fills = Backtest(Market.sample(), ImbalanceBuyer()).run().n_fills
```

</details>

## Fin de los auxiliares

Vuelve al cuaderno principal cuando quieras.